In [5]:
# Cell 1: Environment Setup
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv(override=True)

# Verify API key is loaded
api_key = os.getenv("ANTHROPIC_API_KEY")
if not api_key:
    raise ValueError("ANTHROPIC_API_KEY not found in .env file")
if not api_key.startswith("sk-ant-"):
    raise ValueError("Invalid ANTHROPIC_API_KEY format. Should start with 'sk-ant-'")

print("✓ API Key loaded successfully")
print(f"Key starts with: {api_key[:10]}...")

✓ API Key loaded successfully
Key starts with: sk-ant-api...


In [6]:
# Cell 2: Import Required Libraries
import pandas as pd
import json
import re
import requests
from anthropic import Anthropic

# Initialize Anthropic client
client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
print("✓ Anthropic client initialized")

✓ Anthropic client initialized


In [7]:
# Cell 3: Load Data
df = pd.read_csv("../data/milestone1_output_sathsreehari_k.csv")
print(f"Loaded {len(df)} contracts")
df.head(2)

Loaded 125 contracts


,contract_id,extracted_text,apr,term_months,monthly_payment,penalty
0,1,This contract between Prime Auto and Emily Sto...,10.49,24,959,none
1,2,This contract between ABC Motors and Priya Sha...,3.78,24,804,early termination fee $300


In [8]:
# Cell 4: Define SLA Extraction Prompt
SLA_PROMPT_TEMPLATE = """
You are a legal contract analysis assistant specialized in car lease agreements.

Extract the following SLA details from the contract text.
If a value is missing, return null.
Return ONLY valid JSON. No explanation, no markdown formatting.

Fields to extract:
- interest_rate_apr (number)
- lease_term_months (integer)
- monthly_payment (number)
- down_payment (number or null)
- residual_value (number or null)
- mileage_allowance (number or null)
- overage_charge (number or null)
- early_termination (string or null)
- purchase_option (string or null)
- maintenance_responsibility (string or null)
- warranty_insurance (string or null)
- penalties (string or null)
- missing_or_ambiguous_clauses (string or null)

Contract Text:
\"\"\"
{contract_text}
\"\"\"

Return only the JSON object, nothing else.
"""

In [9]:
# Cell 5: SLA Extraction Function
def extract_sla_with_claude(contract_text: str):
    """
    Extract SLA details from contract text using Claude API
    """
    prompt = SLA_PROMPT_TEMPLATE.format(contract_text=contract_text)
    
    try:
        response = client.messages.create(
            model="claude-3-haiku-20240307",
            max_tokens=1000,
            temperature=0,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        
        raw_text = response.content[0].text.strip()
        
        # Remove markdown code blocks if present
        raw_text = re.sub(r'^```json\s*', '', raw_text)
        raw_text = re.sub(r'\s*```$', '', raw_text)
        
        # Extract JSON
        match = re.search(r"\{[\s\S]*\}", raw_text)
        if not match:
            return {
                "error": "No JSON found",
                "raw_output": raw_text
            }
        
        return json.loads(match.group(0))
        
    except Exception as e:
        return {
            "error": str(e),
            "raw_output": raw_text if 'raw_text' in locals() else None
        }

In [10]:
# Cell 6: Test Single Contract
print("Testing SLA extraction on first contract...")
test_output = extract_sla_with_claude(df.loc[0, "extracted_text"])

if "error" in test_output:
    print(f"❌ Error: {test_output['error']}")
else:
    print("✓ Successfully extracted SLA")
    print(json.dumps(test_output, indent=2))

Testing SLA extraction on first contract...
❌ Error: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CXxfDYbcXFmKWk4qqgm19'}


In [11]:
# Cell 7: Process All Contracts (or subset)
# Start with a small sample to test
sample_size = 5  # Change to len(df) for all contracts
sample_df = df.head(sample_size)

sla_json_records = []

for idx, row in sample_df.iterrows():
    print(f"Processing contract {idx + 1}/{sample_size}...", end=" ")
    
    sla_data = extract_sla_with_claude(row["extracted_text"])
    
    sla_json_records.append({
        "contract_id": int(idx),
        "sla": sla_data
    })
    
    print("✓")

print(f"\n✓ Processed {len(sla_json_records)} contracts")

Processing contract 1/5... ✓
Processing contract 2/5... ✓
Processing contract 3/5... ✓
Processing contract 4/5... ✓
Processing contract 5/5... ✓

✓ Processed 5 contracts


In [12]:
# Cell 8: Save SLA Output
output_file = "../data/milestone2_sla_output.json"
with open(output_file, "w") as f:
    json.dump(sla_json_records, f, indent=2)

print(f"✓ SLA JSON saved to {output_file}")

✓ SLA JSON saved to ../data/milestone2_sla_output.json


In [13]:
# Cell 9: VIN Lookup Functions
def fetch_vehicle_details_from_vin(vin):
    """
    Fetch vehicle make, model, year using NHTSA VIN Decode API
    """
    url = f"https://vpic.nhtsa.dot.gov/api/vehicles/DecodeVinValuesExtended/{vin}?format=json"
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        if not data.get("Results"):
            return None
        
        result = data["Results"][0]
        
        return {
            "vin": vin,
            "make": result.get("Make"),
            "model": result.get("Model"),
            "year": result.get("ModelYear")
        }
    except Exception as e:
        print(f"Error fetching VIN data: {e}")
        return None

In [14]:
# Cell 10: Recall Lookup Function
def fetch_vehicle_recalls(make, model, year):
    """
    Fetch recall information using NHTSA Recall API
    """
    if not make or not model or not year:
        return []
    
    url = (
        "https://api.nhtsa.gov/recalls/recallsByVehicle"
        f"?make={make}&model={model}&modelYear={year}"
    )
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        recalls = data.get("results", [])
        
        return [
            {
                "campaign_number": r.get("NHTSACampaignNumber"),
                "summary": r.get("Summary"),
                "consequence": r.get("Consequence")
            }
            for r in recalls
        ]
    except Exception as e:
        print(f"Error fetching recalls: {e}")
        return []